# Source Coverage Benchmark

Compare one-call SourceCoverage in **serial** and **async** modes, then compare it
against a benchmark-only **DAG / two-stage** SourceCoverage approach. Tokens are not
reported here (not cleanly available yet — a separate production change).

## 1. Setup

Load the CSV and build the judge and framework (one-call `SourceCoverageEvaluator`).

In [ ]:
import time
import json
import asyncio

import pandas as pd

from idp_eval import (
    EvaluationCase,
    EvaluationFramework,
    SourceCoverageEvaluator,
    create_judge,
)

GT_COLUMN = "gt_source_coverage"  # ground-truth column, if present

df = pd.read_csv("golden_set_augmented_tagged.csv").fillna("")

judge = create_judge(verify_ssl=False)

framework = EvaluationFramework(
    evaluators=[SourceCoverageEvaluator(judge, verbose=False)],
    judge=judge,
)


## 2. Serial One-Call Source Coverage

Evaluate every case one at a time with `framework.evaluate`.

In [ ]:
rows = []
start = time.perf_counter()

for i, row in df.iterrows():
    case = EvaluationCase(
        context={
            "theme_business_needs": row.get("theme_businessNeeds", ""),
            "theme_description": row.get("theme_description", ""),
        },
        output={
            "epic_description": row.get("epic_description", ""),
            "epic_success_criteria": row.get("epic_successCriteria", ""),
        },
        case_id=str(row.get("epic_key", i)),
    )
    result = framework.evaluate(case)["source_coverage"]
    details = result.details or {}
    rows.append({
        "case_id": case.case_id,
        "score": result.score,
        "percentage": round(result.score * 100, 1) if result.score is not None else None,
        "label": result.label,
        "explanation": result.explanation,
        "item_count": details.get("final_item_count"),
        "judge_calls": details.get("judge_call_count"),
        "total_ms": details.get("total_ms"),
    })

wall_time_serial = time.perf_counter() - start
df_one_call_serial = pd.DataFrame(rows)

print(f"Serial wall time: {wall_time_serial:.2f}s")
display(df_one_call_serial)


## 3. Async One-Call Source Coverage

Same evaluator, all cases concurrently with `a_evaluate_many(max_concurrency=4)`.

In [ ]:
cases = []
for i, row in df.iterrows():
    cases.append(
        EvaluationCase(
            context={
                "theme_business_needs": row.get("theme_businessNeeds", ""),
                "theme_description": row.get("theme_description", ""),
            },
            output={
                "epic_description": row.get("epic_description", ""),
                "epic_success_criteria": row.get("epic_successCriteria", ""),
            },
            case_id=str(row.get("epic_key", i)),
        )
    )

start = time.perf_counter()
results_async = await framework.a_evaluate_many(cases, max_concurrency=4)
wall_time_async = time.perf_counter() - start

rows = []
for case, result_map in zip(cases, results_async):
    result = result_map["source_coverage"]
    details = result.details or {}
    rows.append({
        "case_id": case.case_id,
        "score": result.score,
        "percentage": round(result.score * 100, 1) if result.score is not None else None,
        "label": result.label,
        "explanation": result.explanation,
        "item_count": details.get("final_item_count"),
        "judge_calls": details.get("judge_call_count"),
        "total_ms": details.get("total_ms"),
    })

df_one_call_async = pd.DataFrame(rows)

print(f"Async wall time: {wall_time_async:.2f}s")
display(df_one_call_async)


## Direct Azure vs Corporate Gateway

Run the exact same SourceCoverage case through both judge paths to isolate gateway
timeout vs model/prompt latency. Diagnostic only. (`AzureOpenAIJudge` and
`PROXY_URL` come from your existing session/config.)

In [ ]:
row = df.iloc[0]
case = EvaluationCase(
    context={
        "theme_business_needs": row.get("theme_businessNeeds", ""),
        "theme_description": row.get("theme_description", ""),
    },
    output={
        "epic_description": row.get("epic_description", ""),
        "epic_success_criteria": row.get("epic_successCriteria", ""),
    },
    case_id=str(row.get("epic_key", 0)),
)

# A. corporate gateway (reuse the existing framework/judge)
start = time.perf_counter()
try:
    gateway_result = framework.evaluate(case)["source_coverage"]
    gateway_elapsed = time.perf_counter() - start
    print(f"Gateway wall time: {gateway_elapsed:.2f}s")
    print("Gateway score:", gateway_result.score)
    print("Gateway label:", gateway_result.label)
except Exception as exc:
    gateway_elapsed = time.perf_counter() - start
    gateway_result = None
    print(f"Gateway failed after: {gateway_elapsed:.2f}s")
    print(type(exc).__name__, str(exc))

# B. direct Azure (reuse the existing notebook-local AzureOpenAIJudge)
azure_judge = AzureOpenAIJudge(timeout=90.0, verify_ssl=False, proxy_url=PROXY_URL)
azure_framework = EvaluationFramework(
    evaluators=[SourceCoverageEvaluator(azure_judge, verbose=False)],
    judge=azure_judge,
)

start = time.perf_counter()
try:
    azure_result = azure_framework.evaluate(case)["source_coverage"]
    azure_elapsed = time.perf_counter() - start
    print(f"Direct Azure wall time: {azure_elapsed:.2f}s")
    print("Direct Azure score:", azure_result.score)
    print("Direct Azure label:", azure_result.label)
    print("Direct Azure explanation:", azure_result.explanation)
except Exception as exc:
    azure_elapsed = time.perf_counter() - start
    azure_result = None
    print(f"Direct Azure failed after: {azure_elapsed:.2f}s")
    print(type(exc).__name__, str(exc))

print("Azure call events:", getattr(azure_judge, "call_events", None))

df_gateway_vs_azure = pd.DataFrame([
    {
        "path": "corporate_gateway",
        "success": gateway_result is not None,
        "wall_time_s": round(gateway_elapsed, 2),
        "score": gateway_result.score if gateway_result else None,
        "label": gateway_result.label if gateway_result else None,
    },
    {
        "path": "direct_azure",
        "success": azure_result is not None,
        "wall_time_s": round(azure_elapsed, 2),
        "score": azure_result.score if azure_result else None,
        "label": azure_result.label if azure_result else None,
    },
])
display(df_gateway_vs_azure)


### How to read the result

- **Direct Azure much faster than gateway** — the gateway path/overhead is likely the
  main bottleneck.
- **Direct Azure succeeds but takes >60s** — the model/prompt is still slow; the
  gateway timeout is simply cutting it off before completion.
- **Direct Azure also fails / approaches 90s** — the SourceCoverage request itself is
  too heavy.
- **Both complete quickly** — investigate intermittent gateway/load behavior.

Actual elapsed time is what matters — do not conclude direct Azure is faster just
because it succeeds. An Azure `400` → `json_object` schema fallback (visible in
`call_events`) is **not** the same as the corporate `504` timeout.

## 4. DAG / Two-Stage Source Coverage

Benchmark-only. Stage 1 extracts important source items from **context only** (never
the output); Stage 2 classifies each fixed item against the output. Two judge calls
per case; score is the Python mean of covered=1.0 / partial=0.5 / missing=0.0.

In [ ]:
from idp_eval.rendering import render_value
from idp_eval.scoring import (
    calculate_coverage,
    coverage_label,
    coverage_status_from_binary,
    coverage_status_score,
)

EXTRACT_SYSTEM = """\
Identify every materially distinct important item in the SOURCE (CONTEXT) that a
faithful representation should preserve. You are given ONLY the source; you are NOT
given the generated answer and must not grade, classify, or compare anything.

SOURCE ITEM RULES
- Preserve every materially distinct important fact, obligation, decision, required
  capability, constraint, prohibition, dependency, measurable target, actor, and
  expected outcome.
- Preserve material qualifiers, including numbers, percentages, limits, thresholds,
  conditions, timing, actors/roles, prohibitions, and dependencies.
- Semantically consolidate repeated or dependent statements. Merge duplicate or
  equivalent statements and merge qualifiers into their underlying item.
- Consolidation is not summarization: never drop a materially distinct important
  item to shorten the list, and do not impose a maximum item count.
- Keep independently satisfiable or independently violatable items separate.
- Do not emit a broad umbrella item in addition to specific items that already fully
  represent it.
- Exclude repetition, examples, filler, incidental prose, and background that does
  not introduce materially important information.
- Never invent source content.

Return only the source item strings."""

CLASSIFY_SYSTEM = (
    "Classify how completely the OUTPUT represents each of the fixed SOURCE ITEMS. "
    "Classify exactly those ids, once each; do not add, remove, or rewrite them. For "
    "each item return meaningfully_present (any meaningful part represented) and "
    "fully_present (full meaning incl. qualifiers represented); if not meaningful, not "
    "full. Judge meaning, not wording. Return only id and the two booleans."
)
EXTRACT_SCHEMA = {"type": "object", "properties": {"source_items": {"type": "array",
    "items": {"type": "object", "properties": {"source_item": {"type": "string"}},
              "required": ["source_item"]}}}, "required": ["source_items"]}
CLASSIFY_SCHEMA = {"type": "object", "properties": {"items": {"type": "array",
    "items": {"type": "object", "properties": {"id": {"type": "string"},
        "meaningfully_present": {"type": "boolean"}, "fully_present": {"type": "boolean"}},
        "required": ["id", "meaningfully_present", "fully_present"]}}}, "required": ["items"]}


async def extract_source_items(case):
    prompt = [{"role": "system", "content": EXTRACT_SYSTEM},
              {"role": "user", "content": "[SOURCE]\n" + render_value(case.context)}]
    resp = await asyncio.to_thread(judge.generate_object, prompt=prompt, schema=EXTRACT_SCHEMA)
    items, seen = [], set()
    for raw in resp.get("source_items", []):
        text = (raw or {}).get("source_item", "").strip()
        key = " ".join(text.lower().split())
        if text and key not in seen:
            seen.add(key)
            items.append({"id": f"s{len(items) + 1}", "source_item": text})
    return items


async def classify_source_items(items, case):
    payload = [{"id": it["id"], "source_item": it["source_item"]} for it in items]
    user = "[SOURCE ITEMS]\n" + json.dumps(payload) + "\n\n[OUTPUT]\n" + render_value(case.output)
    prompt = [{"role": "system", "content": CLASSIFY_SYSTEM}, {"role": "user", "content": user}]
    resp = await asyncio.to_thread(judge.generate_object, prompt=prompt, schema=CLASSIFY_SCHEMA)
    by_id = {a["id"]: a for a in resp.get("items", [])}
    scored = []
    for it in items:
        a = by_id[it["id"]]
        status = coverage_status_from_binary(a["meaningfully_present"], a["fully_present"])
        scored.append({"status": status, "score": coverage_status_score(status)})
    return scored


sem = asyncio.Semaphore(4)


async def run_dag(case):
    async with sem:
        t0 = time.perf_counter()
        items = await extract_source_items(case)
        t1 = time.perf_counter()
        scored = await classify_source_items(items, case) if items else []
        t2 = time.perf_counter()
    score = calculate_coverage(scored) if scored else None
    return {
        "case_id": case.case_id,
        "score": score,
        "percentage": round(score * 100, 1) if score is not None else None,
        "label": coverage_label(score) if score is not None else "not_applicable",
        "item_count": len(scored),
        "judge_calls": 2 if items else 1,
        "extract_ms": round((t1 - t0) * 1000, 1),
        "classify_ms": round((t2 - t1) * 1000, 1) if items else None,
        "total_ms": round((t2 - t0) * 1000, 1),
    }


start = time.perf_counter()
dag_rows = await asyncio.gather(*(run_dag(c) for c in cases))
wall_time_dag = time.perf_counter() - start
df_dag = pd.DataFrame(dag_rows)

print(f"DAG wall time: {wall_time_dag:.2f}s")
display(df_dag)


## 5. One-Call vs DAG Comparison

A small run-level table and a case-level score comparison. GT columns are added when
`GT_COLUMN` is present.

In [ ]:
gt = None
if GT_COLUMN in df.columns:
    # positional, aligned to case order (case_id can repeat, so do not index by it)
    gt = pd.to_numeric(df[GT_COLUMN], errors="coerce").reset_index(drop=True)
    gt = gt.where(gt <= 1.0, gt / 100.0)

comparison = []
for mode, wall, dfm in [
    ("one_call_serial", wall_time_serial, df_one_call_serial),
    ("one_call_async", wall_time_async, df_one_call_async),
    ("dag_async", wall_time_dag, df_dag),
]:
    entry = {
        "mode": mode,
        "wall_time_s": round(wall, 2),
        "mean_score": round(dfm["score"].mean(), 4),
        "mean_latency_ms": round(dfm["total_ms"].mean(), 1),
        "total_judge_calls": int(dfm["judge_calls"].sum()),
    }
    if gt is not None:
        err = dfm["score"].reset_index(drop=True) - gt  # positional (case order)
        entry["MAE"] = round(err.abs().mean(), 4)
        entry["bias"] = round(err.mean(), 4)
    comparison.append(entry)

df_comparison = pd.DataFrame(comparison)
display(df_comparison)

df_score_comparison = pd.DataFrame({
    "case_id": df_one_call_async["case_id"].values,
    "one_call_score": df_one_call_async["score"].values,
    "dag_score": df_dag["score"].values,  # same case order (positional)
})
if gt is not None:
    df_score_comparison["gt_score"] = gt.values
    df_score_comparison["one_call_abs_error"] = (df_score_comparison["one_call_score"] - df_score_comparison["gt_score"]).abs()
    df_score_comparison["dag_abs_error"] = (df_score_comparison["dag_score"] - df_score_comparison["gt_score"]).abs()
display(df_score_comparison)


## 6. Stability and GT Closeness

Run one-call async and DAG async multiple times over the same dataset, then compare
run-to-run stability and error against GT. Uses the same cases, rubric, and scoring;
serial is not part of this experiment.

In [ ]:
N_RUNS = 3

# Unique benchmark id per row (case_id can repeat across rows); align by position.
benchmark_ids = [f"{c.case_id}:{i}" for i, c in enumerate(cases)]

gt_values = (pd.to_numeric(df[GT_COLUMN], errors="coerce").tolist()
             if GT_COLUMN in df.columns else [None] * len(cases))


def _norm_gt(v):
    if v is None or pd.isna(v):
        return None
    v = float(v)
    return v / 100 if v > 1.0 else v  # 0-100 -> 0-1; already 0-1 used directly


gt_by_bid = {benchmark_ids[i]: _norm_gt(gt_values[i]) for i in range(len(cases))}


async def run_one_call_once(run_number):
    start = time.perf_counter()
    results = await framework.a_evaluate_many(cases, max_concurrency=4)
    wall = time.perf_counter() - start
    rows = []
    for i, (c, rm) in enumerate(zip(cases, results)):
        r = rm["source_coverage"]
        d = r.details or {}
        rows.append({"benchmark_id": benchmark_ids[i], "case_id": c.case_id,
                     "run": run_number, "architecture": "one_call", "score": r.score,
                     "item_count": d.get("final_item_count"),
                     "judge_calls": d.get("judge_call_count"), "total_ms": d.get("total_ms")})
    return rows, wall


async def run_dag_once(run_number):
    start = time.perf_counter()
    dag_out = await asyncio.gather(*(run_dag(c) for c in cases))
    wall = time.perf_counter() - start
    rows = []
    for i, (c, dr) in enumerate(zip(cases, dag_out)):
        rows.append({"benchmark_id": benchmark_ids[i], "case_id": c.case_id,
                     "run": run_number, "architecture": "dag", "score": dr["score"],
                     "item_count": dr["item_count"], "judge_calls": dr["judge_calls"],
                     "total_ms": dr["total_ms"]})
    return rows, wall


run_rows, run_summary_rows, failures = [], [], []
for run in range(1, N_RUNS + 1):
    for arch, runner in [("one_call", run_one_call_once), ("dag", run_dag_once)]:
        try:
            rows, wall = await runner(run)
        except Exception as exc:  # no retry; record and continue
            failures.append({"architecture": arch, "run": run, "error": f"{type(exc).__name__}: {exc}"})
            print(f"{arch} run {run} FAILED: {type(exc).__name__}: {exc}")
            continue
        for r in rows:
            gt = gt_by_bid.get(r["benchmark_id"])
            r["gt_score"] = gt
            r["signed_error"] = (r["score"] - gt) if (gt is not None and r["score"] is not None) else None
            r["abs_error"] = abs(r["signed_error"]) if r["signed_error"] is not None else None
        run_rows.extend(rows)
        run_summary_rows.append({
            "architecture": arch, "run": run, "wall_time_s": round(wall, 2),
            "total_judge_calls": int(sum(r["judge_calls"] or 0 for r in rows)),
            "mean_case_ms": round(sum(r["total_ms"] or 0 for r in rows) / len(rows), 1) if rows else None,
        })

df_stability_runs = pd.DataFrame(run_rows)
df_run_summary = pd.DataFrame(run_summary_rows)

# judge-call checks (do not crash the notebook)
oc_bad = df_stability_runs[(df_stability_runs.architecture == "one_call")
                           & df_stability_runs.score.notna() & (df_stability_runs.judge_calls != 1)]
dag_bad = df_stability_runs[(df_stability_runs.architecture == "dag")
                            & df_stability_runs.score.notna() & (df_stability_runs.judge_calls != 2)]
if len(oc_bad):
    print("one_call judge_calls != 1:", list(oc_bad.benchmark_id.unique()))
if len(dag_bad):
    print("dag judge_calls != 2 (score present):", list(dag_bad.benchmark_id.unique()))
if failures:
    print("run failures:", failures)

display(df_run_summary)


In [ ]:
# per-case stability (group the 3 runs by case + architecture)
by = df_stability_runs.groupby(["benchmark_id", "architecture"]).agg(
    case_id=("case_id", "first"), gt_score=("gt_score", "first"),
    mean_score=("score", "mean"), std_score=("score", "std"),
    min_score=("score", "min"), max_score=("score", "max"),
    mean_item_count=("item_count", "mean"), std_item_count=("item_count", "std"),
    min_item_count=("item_count", "min"), max_item_count=("item_count", "max"),
).reset_index()
by["score_range"] = by["max_score"] - by["min_score"]
by["item_count_range"] = by["max_item_count"] - by["min_item_count"]
by["mean_signed_error"] = by["mean_score"] - by["gt_score"]
by["mean_abs_error"] = by["mean_signed_error"].abs()
df_stability_by_case = by

# architecture-level stability + runtime
stab = df_stability_by_case.groupby("architecture").agg(
    cases=("benchmark_id", "nunique"),
    mean_score_std=("std_score", "mean"), median_score_std=("std_score", "median"),
    mean_score_range=("score_range", "mean"), max_score_range=("score_range", "max"),
    mean_item_count_std=("std_item_count", "mean"),
    mean_item_count_range=("item_count_range", "mean"),
    max_item_count_range=("item_count_range", "max"),
).reset_index()
stab["runs"] = N_RUNS
rs = df_run_summary.groupby("architecture").agg(
    mean_wall_time_s=("wall_time_s", "mean"),
    total_judge_calls=("total_judge_calls", "sum"),
    mean_case_latency_ms=("mean_case_ms", "mean"),
).reset_index()
df_stability_summary = stab.merge(rs, on="architecture")[[
    "architecture", "cases", "runs", "mean_score_std", "median_score_std",
    "mean_score_range", "max_score_range", "mean_item_count_std",
    "mean_item_count_range", "max_item_count_range", "mean_wall_time_s",
    "total_judge_calls", "mean_case_latency_ms"]].round(4)

# GT closeness using each case's 3-run MEAN score (not individual runs)
gt_rows = []
for arch, sub in df_stability_by_case.groupby("architecture"):
    s = sub.dropna(subset=["mean_score", "gt_score"])
    err = s["mean_score"] - s["gt_score"]
    gt_rows.append({"architecture": arch, "MAE": err.abs().mean(), "bias": err.mean(),
                    "max_abs_error": err.abs().max(), "RMSE": (err ** 2).mean() ** 0.5})
df_gt_closeness = pd.DataFrame(gt_rows).round(4)

# case-level side-by-side
oc = df_stability_by_case[df_stability_by_case.architecture == "one_call"].set_index("benchmark_id")
dg = df_stability_by_case[df_stability_by_case.architecture == "dag"].set_index("benchmark_id")
rows = []
for bid in oc.index:
    o = oc.loc[bid]
    d = dg.loc[bid] if bid in dg.index else None
    rows.append({
        "benchmark_id": bid, "case_id": o["case_id"], "gt_score": o["gt_score"],
        "one_call_mean_score": o["mean_score"], "one_call_std": o["std_score"],
        "one_call_range": o["score_range"], "one_call_item_count_mean": o["mean_item_count"],
        "one_call_item_count_range": o["item_count_range"], "one_call_abs_error": o["mean_abs_error"],
        "dag_mean_score": d["mean_score"] if d is not None else None,
        "dag_std": d["std_score"] if d is not None else None,
        "dag_range": d["score_range"] if d is not None else None,
        "dag_item_count_mean": d["mean_item_count"] if d is not None else None,
        "dag_item_count_range": d["item_count_range"] if d is not None else None,
        "dag_abs_error": d["mean_abs_error"] if d is not None else None,
    })
df_case_comparison = pd.DataFrame(rows)
df_case_comparison["difference_in_abs_error"] = (
    df_case_comparison["one_call_abs_error"] - df_case_comparison["dag_abs_error"])

# where the two architectures disagree most
tmp = df_case_comparison.copy()
tmp["score_gap"] = (tmp["one_call_mean_score"] - tmp["dag_mean_score"]).abs()
df_largest_disagreements = tmp.sort_values("score_gap", ascending=False).head(10)[[
    "benchmark_id", "case_id", "gt_score", "one_call_mean_score", "dag_mean_score",
    "one_call_std", "dag_std", "one_call_abs_error", "dag_abs_error"]]

display(df_stability_summary)
display(df_gt_closeness)
display(df_case_comparison)
display(df_largest_disagreements)


### How to read stability and GT closeness

- Lower **score std / range** = more stable scoring across runs.
- Lower **item-count std / range** = more stable denominator construction.
- Lower **MAE / RMSE** = closer to GT (using each case's 3-run mean score).
- **bias** near zero = less systematic over/under-scoring.
- `difference_in_abs_error < 0` means one-call is closer to GT for that case; `> 0`
  means DAG is closer. `df_largest_disagreements` shows the cases where the
  architecture choice matters most.